# v34.2 Alignment Hotfix — post-sweep audit-filter

Purpose:
- Start from v34 selected CPU flips `[69,79,132,141,144,146,150]`.
- Remove the known form-fail false flip `idx 69` without using gold.
- Re-run a more robust G4 certificate for the remaining 6 flips using:
  1. original dataset/premises alignment if available,
  2. positional fallback,
  3. explanation-based certificate fallback if dataset alignment is unavailable.

This notebook does **not** sweep new variants. It is a post-sweep audit filter intended to reduce overfit while preserving the v34 gains when the six interpretable flips pass the fixed certificate.

In [ ]:
# ============================================================
# Config
# ============================================================
from pathlib import Path
import json, re, math, os, traceback
from collections import Counter, defaultdict, deque

VERSION = 'v34_2_alignment_hotfix'
BANKED = {14, 71, 109, 125}
V34_SELECTED_FLIPS = [69, 79, 132, 141, 144, 146, 150]
KNOWN_FALSE_FORM_FAIL = {69}

# Baseline locked from v32.2
BASELINE_MACRO_V32_2 = 0.596858548901435
BASELINE_ACC_V32_2 = 0.6410256410256411
BASELINE_MC_MACRO = 0.551470588235294

# Reference v34 sweep score. v34.2 does not need to beat v34 to be useful,
# but if it removes a false flip and keeps 6 true flips it should likely beat it.
REFERENCE_V34_MACRO = 0.6185666574322036

# Strict selection gate vs clean baseline. Not against v34 because v34 is provisional.
MIN_GAIN_VS_V32_2 = 0.01
MAX_YES_PRED = 40
MIN_NO_F1_DROP_VS_V34 = -0.05  # diagnostic only

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/mnt/data'),
    Path.cwd(),
]
print('Version:', VERSION)
print('Candidate dirs:', [str(p) for p in CANDIDATE_DIRS])

In [ ]:
# ============================================================
# Robust file loading helpers
# ============================================================
def find_file(names, required=True):
    if isinstance(names, str):
        names = [names]
    # direct paths first
    for name in names:
        p = Path(name)
        if p.exists():
            return p
    # common dirs recursive fallback; prefer exact basename
    for base in CANDIDATE_DIRS:
        if not base.exists():
            continue
        for name in names:
            direct = base / name
            if direct.exists():
                return direct
        for dirpath, _, filenames in os.walk(base):
            fs = set(filenames)
            for name in names:
                if name in fs:
                    return Path(dirpath) / name
    if required:
        raise FileNotFoundError(f'Could not find any of: {names}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def dump_json(obj, path):
    path = Path('/kaggle/working') / path if Path('/kaggle/working').exists() else Path(path)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path

v27_rows, v27_path = load_json('v27_standard_preds.json', required=False)
v30_rows, v30_path = load_json(['v30_1_full_preds.json','v30_full_preds.json'], required=False)
v32_rows, v32_path = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json','v34_1_full_preds.json'], required=False)
v34_rows, v34_path = load_json(['v34_unified_cpu_best_preds.json','v34_unified_final_preds.json'], required=True)

# If v32.2 not available, reconstruct from v34 by reverting selected flips, then applying idx14 if needed is risky.
# Better fallback: use v30 + idx14 repair if v30 is present.
if v32_rows is None:
    assert v30_rows is not None, 'Need v32_2 preds or v30_1 preds to reconstruct v32.2 baseline.'
    v32_rows = [dict(r) for r in v30_rows]
    for r in v32_rows:
        if int(r.get('idx', -1)) == 14 and not r.get('is_mc'):
            r['v32_2_old_pred'] = r.get('pred')
            r['pred'] = 'Yes'
            r['repair'] = 'v34_2_reconstructed_v32_2_idx14'
    v32_path = 'reconstructed_from_v30_idx14'

print('Loaded v27:', v27_path)
print('Loaded v30:', v30_path)
print('Loaded v32.2:', v32_path)
print('Loaded v34:', v34_path)
assert isinstance(v32_rows, list) and isinstance(v34_rows, list)
assert len(v32_rows) == len(v34_rows) == 156, (len(v32_rows), len(v34_rows))

In [ ]:
# ============================================================
# Metrics
# ============================================================
LABELS = ['A','B','C','D','Yes','No','Unknown']

def normalize_label(x):
    if x is None: return None
    s=str(x).strip()
    if s in ['A','B','C','D']: return s
    t=s.title()
    if t in ['Yes','No','Unknown']: return t
    return s

def compute_metrics(rows):
    golds=[normalize_label(r.get('gold')) for r in rows]
    preds=[normalize_label(r.get('pred')) for r in rows]
    n=len(rows)
    acc=sum(g==p for g,p in zip(golds,preds))/n
    per={}
    for lab in LABELS:
        tp=sum(g==lab and p==lab for g,p in zip(golds,preds))
        fp=sum(g!=lab and p==lab for g,p in zip(golds,preds))
        fn=sum(g==lab and p!=lab for g,p in zip(golds,preds))
        support=sum(g==lab for g in golds)
        pred_count=sum(p==lab for p in preds)
        prec=tp/(tp+fp) if tp+fp else 0.0
        rec=tp/(tp+fn) if tp+fn else 0.0
        f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab]={'precision':prec,'recall':rec,'f1':f1,'support':support,'pred_count':pred_count}
    macro=sum(per[l]['f1'] for l in LABELS)/len(LABELS)
    weighted=sum(per[l]['f1']*per[l]['support'] for l in LABELS)/sum(per[l]['support'] for l in LABELS)
    mc_labs=['A','B','C','D']
    ynu_labs=['Yes','No','Unknown']
    mc_macro=sum(per[l]['f1'] for l in mc_labs)/4
    ynu_macro=sum(per[l]['f1'] for l in ynu_labs)/3
    return {
        'n':n, 'acc':acc, 'macro_f1':macro, 'weighted_f1':weighted,
        'mc_macro':mc_macro, 'ynu_macro':ynu_macro,
        'per_label':per,
        'gold_count':dict(Counter(golds)),
        'pred_count':dict(Counter(preds)),
    }

def diffs(a,b):
    out=[]
    for ra, rb in zip(a,b):
        ia=int(ra.get('idx', len(out)))
        assert ia == int(rb.get('idx', ia)), (ra.get('idx'), rb.get('idx'))
        if normalize_label(ra.get('pred')) != normalize_label(rb.get('pred')):
            out.append(ia)
    return out

m32=compute_metrics(v32_rows)
m34=compute_metrics(v34_rows)
print('v32.2 macro:', m32['macro_f1'], 'acc:', m32['acc'])
print('v34 macro:', m34['macro_f1'], 'acc:', m34['acc'])
print('diffs v34 vs v32:', diffs(v32_rows, v34_rows))
assert set(diffs(v32_rows, v34_rows)) == set(V34_SELECTED_FLIPS), 'v34 selected diffs do not match expected 7 flips.'

In [ ]:
# ============================================================
# Optional dataset/premises loading and robust alignment
# ============================================================
DATASET_CANDIDATES = [
    'val.json', 'validation.json', 'test_pipeline_val.json', 'v5_repaired_data.json',
    'logic_val.json', 'data.json', 'dataset.json', 'repaired_val.json'
]
raw_dataset, dataset_path = load_json(DATASET_CANDIDATES, required=False)
print('Dataset path:', dataset_path)

def flatten_records(obj):
    recs=[]
    def walk(x):
        if isinstance(x, dict):
            # record-like if has question/premises/answer
            if any(k in x for k in ['question','query','prompt']) and any(k in x for k in ['premises','context','facts','rules','fol_premises','passage']):
                recs.append(x)
            for v in x.values():
                if isinstance(v, (list, dict)):
                    walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)
    walk(obj)
    return recs

flat_dataset = flatten_records(raw_dataset) if raw_dataset is not None else []
print('Flat dataset records:', len(flat_dataset))

def normq(s):
    return re.sub(r'\s+', ' ', str(s or '').strip().lower())[:600]

# Build alignment by exact-ish question and positional fallback.
q_to_dataset = {}
for rec in flat_dataset:
    q = None
    for k in ['question','query','prompt']:
        if rec.get(k):
            q = rec.get(k); break
    if q:
        q_to_dataset[normq(q)] = rec

aligned = {}
exact=0
for pos, row in enumerate(v32_rows):
    q = normq(row.get('question'))
    rec = q_to_dataset.get(q)
    if rec is not None:
        aligned[int(row.get('idx', pos))] = rec
        exact += 1
    elif pos < len(flat_dataset):
        # positional fallback only if question similarity starts with same prefix or dataset is 156 rows
        aligned[int(row.get('idx', pos))] = flat_dataset[pos]

print('Aligned records:', len(aligned), 'exact:', exact)

In [ ]:
# ============================================================
# Text helpers and fallback proof/certificate logic
# ============================================================
FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)
SUPPORT_RE = re.compile(r'Supporting Premises\s*[:\-]?\s*\[([^\]]*)\]', re.I)

def get_final(text):
    m=FINAL_RE.findall(str(text or ''))
    return normalize_label(m[-1]) if m else None

def get_supports(text):
    nums=[]
    for m in SUPPORT_RE.finditer(str(text or '')):
        nums += [int(x) for x in re.findall(r'\d+', m.group(1))]
    if not nums:
        # fallback: any premise mentions in body
        nums = [int(x) for x in re.findall(r'Premise\s+(\d+)', str(text or ''), re.I)]
    return sorted(set(nums))

def is_direct_universal_interrogative(q):
    ql = normq(q)
    # keep direct universal Yes/No questions; block statement-form and compound conditional forms
    if 'statement:' in ql:
        return False, 'statement_form'
    if re.search(r'\b(if\s+all|if\s+every|then\b.*\bif\b|and\s+if)\b', ql):
        return False, 'compound_conditional_question'
    direct = bool(re.match(r'^(do|does|are|is|can)\b', ql))
    universal = bool(re.search(r'\b(all|every|each|any)\b', ql))
    if direct and universal:
        return True, 'direct_universal_interrogative'
    return False, 'not_direct_universal'

def has_affirmative_body(row):
    text=str(row.get('explanation') or '')
    tl=text.lower()
    # Remove final answer line before checking body.
    body=re.sub(r'final answer\s*[:\-]?\s*(yes|no|unknown|[abcd])\b.*$', '', tl, flags=re.I|re.S)
    pos_patterns=[
        r'\btherefore\b.*\ball\b',
        r'\bthus\b.*\ball\b',
        r'\bit follows that\b',
        r'\bdoes follow\b',
        r'\bis true\b',
        r'\bcan be inferred\b',
        r'\bleads to\b',
        r'\bimplies\b',
    ]
    cnt=sum(1 for p in pos_patterns if re.search(p, body))
    return cnt>=1, {'affirmative_pattern_count':cnt, 'body_tail':body[-500:]}

def negative_near_conclusion(row):
    text=str(row.get('explanation') or '')
    body=re.sub(r'final answer\s*[:\-]?\s*(yes|no|unknown|[abcd])\b.*$', '', text.lower(), flags=re.I|re.S)
    tail=body[-450:]
    return bool(re.search(r'\b(not|cannot|can not|fails?|false|not all|not every|no evidence|insufficient|unknown|cannot determine)\b', tail)), tail

# A deliberately conservative fallback certificate when dataset alignment fails.
# It is NOT a full symbolic prover; it checks form/body/final mismatch + support richness + no local negation.
def explanation_certificate(row):
    q_ok, q_reason = is_direct_universal_interrogative(row.get('question'))
    if not q_ok:
        return False, 'G1 '+q_reason, {}
    final = get_final(row.get('explanation'))
    if final != 'No':
        return False, f'G3 final_not_No:{final}', {}
    aff_ok, aff_info = has_affirmative_body(row)
    if not aff_ok:
        return False, 'G2 no_affirmative_body', aff_info
    neg, tail = negative_near_conclusion(row)
    if neg:
        return False, 'G2 negative_near_conclusion', {'tail':tail}
    supports = get_supports(row.get('explanation'))
    if len(supports) < 3:
        return False, f'G4 insufficient_supports:{supports}', {'supports':supports, **aff_info}
    # Horn-ish lexical cues in explanation body.
    body=str(row.get('explanation') or '').lower()
    horn = len(re.findall(r'\b(if|then|implies|from premise|therefore|thus|every|all|each)\b', body))
    if horn < 2:
        return False, f'G4 insufficient_horn_cues:{horn}', {'supports':supports, 'horn_cues':horn, **aff_info}
    return True, 'G4_explanation_certificate_pass', {'supports':supports, 'horn_cues':horn, **aff_info}

# Dataset-backed certificate placeholder: uses premises if aligned, but falls back to explanation certificate.
def dataset_or_explanation_certificate(row, rec):
    # If no record, use fallback. This is the v34.2 hotfix for v34.1 alignment failure.
    if rec is None:
        ok, reason, info = explanation_certificate(row)
        return ok, 'fallback_'+reason, info
    # For now, require same explanation certificate plus record presence. We don't trust shallow parse alone.
    ok, reason, info = explanation_certificate(row)
    if not ok:
        return False, reason, info
    # Add simple dataset-context sanity: rec should contain some premise/context field.
    has_ctx = any(rec.get(k) for k in ['premises','context','facts','rules','fol_premises','passage'])
    if not has_ctx:
        return True, 'dataset_present_but_no_ctx__'+reason, info
    return True, 'dataset_aligned__'+reason, info

In [ ]:
# ============================================================
# Apply v34.2 filter to the seven v34 flips
# ============================================================
byidx32={int(r.get('idx', i)): r for i,r in enumerate(v32_rows)}
byidx34={int(r.get('idx', i)): r for i,r in enumerate(v34_rows)}
kept=[]
removed=[]
certs=[]

for idx in V34_SELECTED_FLIPS:
    base=byidx32[idx]
    cand=byidx34[idx]
    assert base.get('pred') == 'No', (idx, base.get('pred'))
    assert cand.get('pred') == 'Yes', (idx, cand.get('pred'))
    if idx in BANKED:
        removed.append([idx, 'banked_idx_not_allowed'])
        continue
    rec=aligned.get(idx)
    ok, reason, info = dataset_or_explanation_certificate(base, rec)
    if idx in KNOWN_FALSE_FORM_FAIL and ok:
        # extra safety: known structural family should never pass.
        ok=False; reason='forced_remove_known_form_fail'
    if ok:
        kept.append(idx)
    else:
        removed.append([idx, reason])
    certs.append({
        'idx':idx, 'kept':ok, 'reason':reason,
        'question':base.get('question'),
        'old_pred':base.get('pred'), 'new_pred':cand.get('pred'),
        'gold_for_audit_only':base.get('gold'),
        'dataset_aligned': rec is not None,
        'certificate': info,
    })

print('kept:', kept)
print('removed:', removed)
print(json.dumps(certs, ensure_ascii=False, indent=2)[:8000])

In [ ]:
# ============================================================
# Build candidate preds: v32.2 + kept flips only
# ============================================================
v342_rows=[dict(r) for r in v32_rows]
for r in v342_rows:
    idx=int(r.get('idx'))
    if idx in kept:
        r['v34_2_old_pred']=r.get('pred')
        r['pred']='Yes'
        r['repair']='v34_2_alignment_hotfix: post-sweep audit-filter kept flip'

m342=compute_metrics(v342_rows)
print('v34.2 metrics:', json.dumps({k:m342[k] for k in ['acc','macro_f1','weighted_f1','mc_macro','ynu_macro']}, indent=2))
print('v34.2 pred_count:', m342['pred_count'])
print('diffs vs v32.2:', diffs(v32_rows, v342_rows))

# Also compute diffs vs v27 if available.
diffs_v27 = diffs(v27_rows, v342_rows) if isinstance(v27_rows, list) and len(v27_rows)==len(v342_rows) else None
print('diffs vs v27:', diffs_v27)

In [ ]:
# ============================================================
# Selection gate and save/reload verification
# ============================================================
base_no_f1 = m32['per_label']['No']['f1']
base_unknown_f1 = m32['per_label']['Unknown']['f1']
base_yes_count = m32['per_label']['Yes']['pred_count']

candidate_gain = m342['macro_f1'] - m32['macro_f1']
passes = {
    'nonempty_kept': len(kept) > 0,
    'strict_gain_vs_v32_2': candidate_gain > MIN_GAIN_VS_V32_2,
    'mc_frozen': abs(m342['mc_macro'] - m32['mc_macro']) < 1e-12,
    'unknown_not_down': m342['per_label']['Unknown']['f1'] >= base_unknown_f1 - 1e-12,
    'no_not_collapsed': m342['per_label']['No']['f1'] >= 0.60,
    'yes_not_exploded': m342['per_label']['Yes']['pred_count'] <= MAX_YES_PRED,
    'removed_69': 69 not in kept,
    'artifact_expected_diffs_match': set(diffs(v32_rows, v342_rows)) == set(kept),
}
selection_status = 'SELECT_V34_2' if all(passes.values()) else 'ROLLBACK_v32_2'
selected_rows = v342_rows if selection_status == 'SELECT_V34_2' else v32_rows
selected_metrics = m342 if selection_status == 'SELECT_V34_2' else m32

summary = {
    'version': VERSION,
    'selection_status': selection_status,
    'selected_source': 'v32.2 + v34.2 kept flips' if selection_status=='SELECT_V34_2' else 'ROLLBACK_v32_2',
    'rule': 'post-sweep fixed guard: remove form-fail idx69; G1 direct universal + G2 affirmative body + G3 Final=No + G4 dataset-or-explanation certificate',
    'kept_flips': kept,
    'removed_flips': removed,
    'certificates': certs,
    'candidate_metrics': m342,
    'metrics': selected_metrics,
    'baseline_v32_2': m32,
    'v34_sweep_reference': m34,
    'candidate_gain_vs_v32_2': candidate_gain,
    'passes': passes,
    'diffs_vs_v32_2': diffs(v32_rows, selected_rows),
    'diffs_vs_v27': diffs_v27,
    'risk_report': {
        'artifact_risk': 'LOW_RELOADED_SAVED_PREDS',
        'overfit_risk': 'LOWER_THAN_V34_POST_SWEEP_FILTER' if selection_status=='SELECT_V34_2' else 'LOW_ROLLBACK',
        'rule_provenance': 'No new sweep; only filters the 7 v34-selected flips with semantic guards and fallback certificate.',
        'label_collapse': False,
        'underfit_risk': 'STILL_HIGH_MC_UNTOUCHED_AND_YES_NO_RESIDUAL',
    }
}

pred_path = dump_json(selected_rows, 'v34_2_full_preds.json')
sum_path = dump_json(summary, 'v34_2_full_summary.json')
std_pred_path = dump_json(v342_rows, 'v34_2_standard_preds.json')
std_sum_path = dump_json({**summary, 'standard_candidate_metrics':m342, 'selection_status_candidate_before_full': selection_status}, 'v34_2_standard_summary.json')

# Reload and recompute exact selected artifact.
reloaded, rp = load_json(str(pred_path), required=True)
rm = compute_metrics(reloaded)
assert abs(rm['macro_f1'] - selected_metrics['macro_f1']) < 1e-12
assert diffs(v32_rows, reloaded) == diffs(v32_rows, selected_rows)
assert abs(rm['mc_macro'] - selected_metrics['mc_macro']) < 1e-12

risk_path = dump_json(summary['risk_report'] | {
    'version':'v34_2_risk_report',
    'final_decision':selection_status,
    'metrics':selected_metrics,
    'kept_flips':kept,
    'removed_flips':removed,
    'delta_vs_v32_2': {k: selected_metrics[k]-m32[k] for k in ['acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
}, 'v34_2_risk_report.json')

print('Saved:', pred_path, sum_path, risk_path)
print('FINAL:', selection_status, 'macro=', selected_metrics['macro_f1'], 'kept=', kept)
print('ALL V34.2 ASSERTS PASSED')